In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [3]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [4]:
from dotenv import load_dotenv
import openai
import os, torch, numpy as np
from utils.call_llm import call_llm
import json
from sklearn.pipeline import make_pipeline

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

## Chain of Thoughts

In [5]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case

case_name_set = ["stereotype"]  # "stereotype", "manipulation"
strategy = "optimized"          # "optimized", "zero_plus"
max_tokens = 700

os.makedirs(f"results/{model_filename}/cot/reasoning", exist_ok=True)

try:
    for case_name in case_name_set:
        print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) ===")

        if case_name.lower() == "manipulation":
            case = manipulation_case
            task_definition = manipulation_definition_short
            data = sample_mentalmanip
        elif case_name.lower() == "stereotype":
            case = stereotypes_case
            task_definition = stereotype_definition_short_binary
            data = sample_mgsd
        else:
            raise ValueError(f"Unknown case name: {case_name}")

        cot_classifier = ChainOfThoughts(
            case=case,
            client=client,
            model=model,
            max_tokens=max_tokens,
            task_definition=task_definition,
            num_reasoning_steps=4,
        )

        rows = []
        detailed_reasoning = []

        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, metrics = cot_classifier.classify_with_strategy(
                    text, strategy=strategy
                )

                mapped_label = case.label_map.get(
                    predicted_label.strip(), list(case.label_map.values())[-1]
                )

                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "raw_pred_label": predicted_label,
                    "max_tokens": cot_classifier.max_tokens,
                    "tokens_used": metrics.get("tokens_used"),
                    "prompt_tokens": metrics.get("prompt_tokens"),
                    "completion_tokens": metrics.get("completion_tokens"),
                    "latency": metrics.get("latency"),
                    "strategy": strategy,
                }
                rows.append(results)

                raw_resp = metrics.get("raw_response", "")
                steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                reasoning_detail = {
                    "sample_id": idx,
                    "raw_response": raw_resp,
                    "parsed_steps": steps,
                    "analysis": analysis,       
                    "final_line": final_line,
                    "final_label": predicted_label,
                    "mapped_label": mapped_label,
                }
                detailed_reasoning.append(reasoning_detail)

            except Exception as e:
                print(f"Error processing sample {idx}: {e}")
                continue

        if not rows:
            print(f"No successful classifications for {case_name}")
            continue

        output_file = f"results/{model_filename}/cot/reasoning/results_{case_name.lower()}_{strategy}_cot.csv"
        reasoning_file = f"results/{model_filename}/cot/reasoning/reasoning_{case_name.lower()}_{strategy}_cot.json"

        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"=== Saved {len(df_out)} rows to {output_file} ===")

        with open(reasoning_file, 'w', encoding='utf-8') as f:
            json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
        print(f"=== Saved detailed reasoning to {reasoning_file} ===")

        try:
            if case_name.lower() == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            elif case_name.lower() == "stereotype":
                y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()

            print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
            print(classification_report(y_true, y_pred, zero_division=0))
            print(f"\n=== Confusion Matrix for {case_name} ===")
            labels = sorted(set(y_true) | set(y_pred))
            print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))

            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")

            print(f"\n=== Label Distribution ===")
            print("True labels:")
            print(pd.Series(y_true).value_counts())
            print("Predicted labels:")
            print(pd.Series(y_pred).value_counts())

        except Exception as e:
            print(f"Error in evaluation for {case_name}: {e}")

        metrics = cot_classifier.get_metrics()
        print(f"\n=== CoT Performance Metrics ===")
        print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
        print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
        print(f"- Total calls: {metrics['total_calls']}")


        print(f"\n=== Sample Reasoning (raw/parsed) ===")
        for i, reasoning in enumerate(detailed_reasoning[:3]):
            print(f"\nSample {reasoning['sample_id']}:")
            print(f"True Label: {data.iloc[reasoning['sample_id']][case.label_col]}")
            print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
            if reasoning['parsed_steps']:
                print("Parsed Steps:")
                for step in reasoning['parsed_steps']:
                    print(f"  Step {step['step']}: {step['content'][:150]}...")
            else:
                print("No explicit 'Step N' blocks found.")
            if reasoning['final_line']:
                print(f"Final Line: {reasoning['final_line']}")
            print("Raw Response (truncated):")
            print((reasoning['raw_response'] or "")[:300] + "...")
            print("-" * 50)

except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")



=== Running Chain of Thought for stereotype (optimized) ===


Processing stereotype: 100%|██████████| 500/500 [24:50<00:00,  2.98s/it]  

=== Saved 500 rows to results/openai_4.1_mini/cot/reasoning/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/reasoning/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.73      0.76      0.75       250
   unrelated       0.75      0.72      0.73       250

    accuracy                           0.74       500
   macro avg       0.74      0.74      0.74       500
weighted avg       0.74      0.74      0.74       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         190         60
unrelated           70        180

=== Accuracy for stereotype: 74.00% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    260
unrelated     240
Name: count, dtype: int64

=== CoT Pe

In [ ]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case

case_name_set = ["stereotype"]  # "stereotype", "manipulation"
strategy = "optimized"          # "optimized", "zero_plus"
max_tokens = 700
role_playing_mode = "passive"
selected_profiles = ["profile1" for i in range(1, 10)]

os.makedirs(f"results/{model_filename}/cot/reasoning", exist_ok=True)

try:
    for profile_n in selected_profiles:
        for case_name in case_name_set:
            print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) ===")
    
            if case_name.lower() == "manipulation":
                case = manipulation_case
                task_definition = manipulation_definition_short
                data = sample_mentalmanip
            elif case_name.lower() == "stereotype":
                case = stereotypes_case
                task_definition = stereotype_definition_short_binary
                data = sample_mgsd
            else:
                raise ValueError(f"Unknown case name: {case_name}")
    
            cot_classifier = ChainOfThoughts(
                case=case,
                client=client,
                model=model,
                max_tokens=max_tokens,
                task_definition=task_definition,
                person_key=profile_n,
                role_playing=role_playing_mode
            )
    
            rows = []
            detailed_reasoning = []
    
            for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
    
                try:
                    predicted_label, metrics = cot_classifier.classify_with_strategy(
                        text, strategy=strategy
                    )
    
                    mapped_label = case.label_map.get(
                        predicted_label.strip(), list(case.label_map.values())[-1]
                    )
    
                    results = {
                        "sample_id": idx,
                        "text": text,
                        "true_label": true_label,
                        "pred_label": mapped_label,
                        "raw_pred_label": predicted_label,
                        "max_tokens": cot_classifier.max_tokens,
                        "tokens_used": metrics.get("tokens_used"),
                        "prompt_tokens": metrics.get("prompt_tokens"),
                        "completion_tokens": metrics.get("completion_tokens"),
                        "latency": metrics.get("latency"),
                        "strategy": strategy,
                    }
                    rows.append(results)
    
                    raw_resp = metrics.get("raw_response", "")
                    steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                    reasoning_detail = {
                        "sample_id": idx,
                        "raw_response": raw_resp,
                        "parsed_steps": steps,
                        "analysis": analysis,       
                        "final_line": final_line,
                        "final_label": predicted_label,
                        "mapped_label": mapped_label,
                    }
                    detailed_reasoning.append(reasoning_detail)
    
                except Exception as e:
                    print(f"Error processing sample {idx}: {e}")
                    continue
    
            if not rows:
                print(f"No successful classifications for {case_name}")
                continue
    
            output_file = f"results/{model_filename}/cot/reasoning/results_{case_name.lower()}_{strategy}_cot.csv"
            reasoning_file = f"results/{model_filename}/cot/reasoning/reasoning_{case_name.lower()}_{strategy}_cot.json"
    
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"=== Saved {len(df_out)} rows to {output_file} ===")
    
            with open(reasoning_file, 'w', encoding='utf-8') as f:
                json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
            print(f"=== Saved detailed reasoning to {reasoning_file} ===")
    

            try:
                if case_name.lower() == "manipulation":
                    y_true = df_out["true_label"].astype(int)
                    y_pred = df_out["pred_label"].astype(int)
                elif case_name.lower() == "stereotype":
                    y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
    
                print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
                print(classification_report(y_true, y_pred, zero_division=0))
                print(f"\n=== Confusion Matrix for {case_name} ===")
                labels = sorted(set(y_true) | set(y_pred))
                print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))
    
                accuracy = (y_true == y_pred).mean()
                print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")
    
                print(f"\n=== Label Distribution ===")
                print("True labels:")
                print(pd.Series(y_true).value_counts())
                print("Predicted labels:")
                print(pd.Series(y_pred).value_counts())
    
            except Exception as e:
                print(f"Error in evaluation for {case_name}: {e}")
    
            metrics = cot_classifier.get_metrics()
            print(f"\n=== CoT Performance Metrics ===")
            print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
            print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
            print(f"- Total calls: {metrics['total_calls']}")
    
    
            print(f"\n=== Sample Reasoning (raw/parsed) ===")
            for i, reasoning in enumerate(detailed_reasoning[:3]):
                print(f"\nSample {reasoning['sample_id']}:")
                print(f"True Label: {data.iloc[reasoning['sample_id']][case.label_col]}")
                print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
                if reasoning['parsed_steps']:
                    print("Parsed Steps:")
                    for step in reasoning['parsed_steps']:
                        print(f"  Step {step['step']}: {step['content'][:150]}...")
                else:
                    print("No explicit 'Step N' blocks found.")
                if reasoning['final_line']:
                    print(f"Final Line: {reasoning['final_line']}")
                print("Raw Response (truncated):")
                print((reasoning['raw_response'] or "")[:300] + "...")
                print("-" * 50)
    
except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")


=== Running Chain of Thought for stereotype (zero_plus) ===


Processing stereotype: 100%|██████████| 500/500 [15:27<00:00,  1.86s/it]  

=== Saved 500 rows to results/openai_4.1_mini/cot/reasoning/results_stereotype_zero_plus_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/reasoning/reasoning_stereotype_zero_plus_cot.json ===

=== Classification Report for stereotype (zero_plus) ===
              precision    recall  f1-score   support

  stereotype       0.73      0.61      0.67       250
   unrelated       0.67      0.78      0.72       250

    accuracy                           0.69       500
   macro avg       0.70      0.69      0.69       500
weighted avg       0.70      0.69      0.69       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         153         97
unrelated           56        194

=== Accuracy for stereotype: 69.40% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
unrelated     291
stereotype    209
Name: count, dtype: int64

=== CoT Pe

## Role playing with Chain-of-Thoughts

In [ ]:
from tqdm import tqdm
import os, json
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short
from manipulation_definitions import manipulation_definition_short
from cases.get_case_config import get_case_config
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from sklearn.metrics import classification_report, confusion_matrix

case_name_set = ["stereotype"]  # , "manipulation" # "stereotype" or "manipulation"
max_tokens = 700
role_playing_modes = "passive"
selected_profiles = ["profile1" for i in range(1, 10)]

os.makedirs(f"results/{model_filename}/cot", exist_ok=True)

try:

    for person_key in selected_profiles:
        for role_playing in role_playing_modes:
            for case_name in case_name_set:
                    print(f"\n=== Running CoT with 700 tokens ===")
            
                    if case_name.lower() == "manipulation":
                        case = manipulation_case
                        task_definition = manipulation_definition_short
                        data = sample_mentalmanip
                    elif case_name.lower() == "stereotype":
                        case = stereotypes_case
                        task_definition = stereotype_definition_short_binary
                        data = sample_mgsd
                    else:
                        raise ValueError(f"Unknown case name: {case_name}")
            
                    cot_classifier = ChainOfThoughts(
                        case=case,
                        client=client,
                        model=model,
                        max_tokens=max_tokens,
                        task_definition=task_definition,
                        num_reasoning_steps=5,
                        person_key=person_key,
                        role_playing=role_playing,
                    )
            
                    rows = []
                    detailed_reasoning = []
            
                    for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {num_steps}steps"):
                        text = row[case["input_col"]]
                        true_label = row[case["label_col"]]
                        if isinstance(true_label, str):
                            true_label = true_label.strip()
            
                        cot_result = cot_classifier.classify_with_reasoning(text)
                        predicted_label = cot_result.final_label
            
                        mapped_label = case["label_map"].get(predicted_label.strip(), list(case["label_map"].values())[-1])
            
                        stats = cot_result.stats
            
                        results = {
                            "sample_id": idx,
                            "text": text,
                            "true_label": true_label,
                            "pred_label": mapped_label,
                            "raw_pred_label": predicted_label,
                            "max_tokens": cot_classifier.max_tokens,
                            "tokens_used": stats.tokens_used if stats else None,
                            "prompt_tokens": stats.prompt_tokens if stats else None,
                            "completion_tokens": stats.completion_tokens if stats else None,
                            "latency": stats.latency if stats else None,
                            "num_reasoning_steps": len(cot_result.reasoning_steps),
                            "confidence": cot_result.confidence,
                            "person_key": cot_result.person_key,
                            "role_playing": cot_result.role_playing,
                        }
            
                        rows.append(results)
            
                        reasoning_detail = {
                            "sample_id": idx,
                            "reasoning_steps": [
                                {"step": step.step_number, "content": step.content}
                                for step in cot_result.reasoning_steps
                            ],
                            "final_reasoning": cot_result.final_reasoning,
                            "final_label": cot_result.final_label,
                            "confidence": cot_result.confidence,
                            "person_key": cot_result.person_key,
                            "role_playing": cot_result.role_playing,
                        }
            
                        detailed_reasoning.append(reasoning_detail)
            
            
                    output_file = f"results/{model_filename}/cot/role_playing/{person_key}_{role_playing}/results_{case_name.lower()}_cot_{num_steps}steps.csv"
                    reasoning_file = f"results/{model_filename}/cot/role_playing/{person_key}_{role_playing}/reasoning_{case_name.lower()}_cot_{num_steps}steps.json"
            
                    df_out = pd.DataFrame(rows)
                    df_out.to_csv(output_file, index=False)
                    print(f"=== Saved {len(df_out)} rows to {output_file} ===")
            
                    with open(reasoning_file, 'w', encoding='utf-8') as f:
                        json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
                    print(f"=== Saved detailed reasoning to {reasoning_file} ===")
            
        
                    if case_name == "manipulation":
                        y_true = df_out["true_label"].astype(int)
                        y_pred = df_out["pred_label"].astype(int)
                    elif case_name == "stereotype":
                        y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                        y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
            
                    print("\n=== Classification Report ===")
                    print(classification_report(y_true, y_pred))
                    print("\n=== Confusion Matrix ===")
                    labels = sorted(set(y_true) | set(y_pred))
                    print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=labels, columns=labels))
            
                    accuracy = (y_true == y_pred).mean()
                    print(f"\n=== Accuracy: {accuracy:.2%} ===")
            
                    metrics = cot_classifier.get_metrics()
                    print(f"\n=== CoT Performance Metrics: ===")
                    print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
                    print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
                    print(f"- Total calls: {metrics['total_calls']}")
            
                    print("\n Sample Reasoning Chains:")
                    for i, reasoning in enumerate(detailed_reasoning[:3]):
                        print(f"\nSample {reasoning['sample_id']}:")
                        print(f"Final Label: {reasoning['final_label']}")
                        if reasoning['confidence'] > 0:
                            print(f"Confidence: {reasoning['confidence']:.2f}")
                        print("Reasoning Steps:")
                        for step in reasoning['reasoning_steps']:
                            print(f"  Step {step['step']}: {step['content'][:100]}...")
                        print(f"Final Reasoning: {reasoning['final_reasoning'][:150]}...")
                        print("-" * 50)

except Exception as e:
    print(f"== Error with steps={num_steps}: {e} ==")
    import traceback
    traceback.print_exc()

print("\n✅ Chain-of-Thoughts Evaluation Complete")